# Notebook 04 — Final Model Training (All 3 Datasets)
Trains COA-ANN and COA-CNN+BiLSTM on train+val combined per dataset.
Evaluates on held-out test set only.

**Inputs:** NB01 + NB02 + NB03 outputs  
**GPU:** MirroredStrategy (both T4s) + mixed precision  
**Time:** ~2-3 hrs


In [1]:
import os, random as _random, warnings, re, string, json, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
MAX_VOCAB  = 50000
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

WORKING     = "/kaggle/working"
SHARED_DIR  = f"{WORKING}/shared_data"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"

def ensure_dirs(dataset=None):
    for base in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
        os.makedirs(base, exist_ok=True)
        if dataset:
            os.makedirs(f"{base}/{dataset}", exist_ok=True)
        else:
            for ds in DATASETS:
                os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
print(f"MirroredStrategy: {N_GPUS} GPU(s)")

ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()
print(f"TF {tf.__version__}")

ensure_dirs()


2026-06-01 14:07:19.504563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780322839.737327      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780322839.797613      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780322840.341963      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780322840.342000      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780322840.342003      23 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
MirroredStrategy: 2 GPU(s)
ANN_BATCH=2048  CNN_BATCH=256
TF 2.19.0


I0000 00:00:1780322854.583939      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780322854.590221      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    idx = np.argmax(f1s[:-1])
    return float(thr[idx])

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.int32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

print("Helpers ready.")


Helpers ready.


In [3]:
def build_ann(h1, h2, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn(filters, lstm_units, lr, vocab_sz, emb_mat, max_len,
               seed=SEED, pretrained=True, use_bilstm=True, use_dropout=True):
    set_seeds_tf(seed)
    with strategy.scope():
        emb = (tf.keras.layers.Embedding(vocab_sz, EMBED_DIM,
                                          weights=[emb_mat], trainable=False)
               if pretrained else
               tf.keras.layers.Embedding(vocab_sz, EMBED_DIM, trainable=True))
        layers = [emb, tf.keras.layers.Conv1D(filters, 3, activation="relu"),
                  tf.keras.layers.MaxPooling1D(2)]
        if use_bilstm:
            layers.append(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)))
        else:
            layers.append(tf.keras.layers.GlobalMaxPooling1D())
        layers.append(tf.keras.layers.Dense(64, activation="relu"))
        if use_dropout:
            layers.append(tf.keras.layers.Dropout(0.5))
        layers.append(tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"))
        m = tf.keras.Sequential(layers)
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def shallow_ann(h1, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Model builders ready.")


Model builders ready.


## 1. Paths

In [4]:
# ── UPDATE: replace username/slug with your actual Kaggle values ──────────────
NB01_DIR    = "/kaggle/input/notebooks/abhaysharma01702/01-preprocessing"
NB01_SHARED = f"{NB01_DIR}/shared_data"

print("Verifying NB01 shared_data...")
for ds in DATASETS:
    d = f"{NB01_SHARED}/{ds}"
    ok = os.path.isdir(d)
    n  = len(os.listdir(d)) if ok else 0
    cfg = json.load(open(f"{d}/config.json")) if ok else {}
    print(f"  {ds}: {'✓' if ok else '✗ MISSING'}  {n} files  "
          f"max_len={cfg.get('max_len','?')}  n_train={cfg.get('n_train','?')}")

NB02_DIR = "/kaggle/input/notebooks/abhaysharma01702/02-coa-ann"
NB03_DIR = "/kaggle/input/notebooks/abhaysharma01702/03-coa-cnn"
NB02_RESULTS = f"{NB02_DIR}/saved_results"
NB03_RESULTS = f"{NB03_DIR}/saved_results"

print("Verifying COA results...")
for ds in DATASETS:
    ann_ok = os.path.exists(f"{NB02_RESULTS}/{ds}/coa_ann.json")
    cnn_ok = os.path.exists(f"{NB03_RESULTS}/{ds}/coa_cnn.json")
    print(f"  {ds}: ANN={'✓' if ann_ok else '✗'}  CNN={'✓' if cnn_ok else '✗'}")


Verifying NB01 shared_data...
  amazon: ✓  23 files  max_len=80  n_train=450000
  yelp: ✓  23 files  max_len=152  n_train=420000
  twitter: ✓  23 files  max_len=50  n_train=360000
Verifying COA results...
  amazon: ANN=✓  CNN=✓
  yelp: ANN=✓  CNN=✓
  twitter: ANN=✓  CNN=✓


## 2. Train Final Models per Dataset

In [5]:
def load_dataset(dataset, nb01_shared):
    """Load all arrays for a given dataset."""
    d_dir = f"{nb01_shared}/{dataset}"
    cfg   = json.load(open(f"{d_dir}/config.json"))
    d = {
        "name"         : dataset,
        "cfg"          : cfg,
        "max_len"      : cfg["max_len"],
        "vocab_size"   : cfg["vocab_size"],
        "X_emb_train"  : np.load(f"{d_dir}/X_emb_train.npy"),
        "X_emb_val"    : np.load(f"{d_dir}/X_emb_val.npy"),
        "X_emb_test"   : np.load(f"{d_dir}/X_emb_test.npy"),
        "X_seq_train"  : np.load(f"{d_dir}/X_seq_train.npy"),
        "X_seq_val"    : np.load(f"{d_dir}/X_seq_val.npy"),
        "X_seq_test"   : np.load(f"{d_dir}/X_seq_test.npy"),
        "y_train"      : np.load(f"{d_dir}/y_train.npy"),
        "y_val"        : np.load(f"{d_dir}/y_val.npy"),
        "y_test"       : np.load(f"{d_dir}/y_test.npy"),
        "emb_matrix"   : np.load(f"{d_dir}/emb_matrix.npy"),
        "class_weights": {int(k):float(v) for k,v in
                          json.load(open(f"{d_dir}/class_weights.json")).items()},
    }
    with open(f"{d_dir}/test_texts_raw.txt", encoding="utf-8") as f:
        d["test_texts_raw"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_train.txt", encoding="utf-8") as f:
        d["X_txt_train"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_val.txt", encoding="utf-8") as f:
        d["X_txt_val"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_test.txt", encoding="utf-8") as f:
        d["X_txt_test"] = f.read().splitlines()
    print(f"  {dataset}: train={len(d['y_train']):,} val={len(d['y_val']):,} "
          f"test={len(d['y_test']):,} max_len={d['max_len']}")
    return d

def tv_combined(d):
    """Train+val combined for final training."""
    return {
        "X_emb": np.vstack([d["X_emb_train"], d["X_emb_val"]]),
        "X_seq": np.vstack([d["X_seq_train"], d["X_seq_val"]]),
        "y"    : np.concatenate([d["y_train"], d["y_val"]]),
        "X_txt": d["X_txt_train"] + d["X_txt_val"],
    }

print("Loader ready.")


Loader ready.


In [6]:
all_results = {}

for ds in DATASETS:
    print(f"\n{'='*60}\nFINAL TRAINING | {ds.upper()}\n{'='*60}")

    d = load_dataset(ds, NB01_SHARED)
    ml = d["max_len"]; vs = d["vocab_size"]; emb = d["emb_matrix"]

    # Load COA hyperparameters for this specific dataset
    ann_cfg = json.load(open(f"{NB02_RESULTS}/{ds}/coa_ann.json"))
    cnn_cfg = json.load(open(f"{NB03_RESULTS}/{ds}/coa_cnn.json"))
    h1  = ann_cfg["best_h1"];  h2  = ann_cfg["best_h2"]; ann_lr = ann_cfg["best_lr"]
    f_  = cnn_cfg["best_filters"]; lstm_u = cnn_cfg["best_lstm_units"]; cnn_lr = cnn_cfg["best_lr"]

    print(f"  COA-ANN   : H1={h1} H2={h2} LR={ann_lr:.8f}  (val_F1={ann_cfg['best_val_f1']})")
    print(f"  COA-CNN   : F={f_} LSTM={lstm_u} LR={cnn_lr:.8f}  (val_F1={cnn_cfg['best_val_f1']})")

    # Train+Val combined
    tv = tv_combined(d)
    cw_tv = dict(zip(
        [int(k) for k in np.unique(tv["y"])],
        [float(v) for v in compute_class_weight("balanced",
                            classes=np.unique(tv["y"]), y=tv["y"])]
    ))
    n_es = int(len(tv["y"])*0.1)

    m_dir = f"{MODELS_DIR}/{ds}"; os.makedirs(m_dir, exist_ok=True)
    r_dir = f"{RESULTS_DIR}/{ds}"; os.makedirs(r_dir, exist_ok=True)
    ds_results = []

    # ── COA-ANN ──────────────────────────────────────────────────────────────
    print(f"\n  Training COA-ANN (batch={ANN_BATCH})...")
    set_seeds_tf()
    model = build_ann(h1, h2, ann_lr)
    ckpt  = f"{m_dir}/coa_ann_best.keras"
    model.fit(
        make_tf_ds(tv["X_emb"][:-n_es], tv["y"][:-n_es], ANN_BATCH),
        validation_data=make_tf_ds(tv["X_emb"][-n_es:], tv["y"][-n_es:],
                                    ANN_BATCH, shuffle=False),
        epochs=100,
        callbacks=[make_es(10,verbose=1),
                   ModelCheckpoint(ckpt,monitor="val_loss",save_best_only=True,verbose=1)],
        class_weight=cw_tv, verbose=1)
    model = tf.keras.models.load_model(ckpt)
    ann_prob = model.predict(
        make_tf_ds(d["X_emb_test"], np.zeros(len(d["y_test"])),
                    ANN_BATCH, shuffle=False), verbose=1).flatten()
    ann_pred = (ann_prob > 0.5).astype(int)
    ann_res  = evaluate("COA-ANN", d["y_test"], ann_pred, ann_prob)
    best_t   = best_threshold(d["y_test"], ann_prob)
    ann_res["optimal_threshold"] = round(best_t,4)
    ann_res["f1_at_optimal"] = round(float(f1_score(d["y_test"],(ann_prob>best_t).astype(int))),4)
    ds_results.append(ann_res)
    print(f"  COA-ANN: F1={ann_res['F1-score']}  MCC={ann_res['MCC']}")

    np.save(f"{r_dir}/ann_coa_prob.npy", ann_prob)
    json.dump(ann_res, open(f"{r_dir}/ann_coa_metrics.json","w"), indent=2)
    model.save(f"{m_dir}/coa_ann_final.keras")
    del model; gc.collect()

    # ── COA-CNN+BiLSTM ────────────────────────────────────────────────────────
    print(f"\n  Training COA-CNN+BiLSTM (batch={CNN_BATCH})...")
    set_seeds_tf()
    model = build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml)
    ckpt  = f"{m_dir}/coa_cnn_best.keras"
    model.fit(
        make_seq_ds(tv["X_seq"][:-n_es], tv["y"][:-n_es], CNN_BATCH),
        validation_data=make_seq_ds(tv["X_seq"][-n_es:], tv["y"][-n_es:],
                                     CNN_BATCH, shuffle=False),
        epochs=30,
        callbacks=[make_es(5,verbose=1),
                   ModelCheckpoint(ckpt,monitor="val_loss",save_best_only=True,verbose=1)],
        class_weight=cw_tv, verbose=1)
    model = tf.keras.models.load_model(ckpt)
    cnn_prob = model.predict(
        make_seq_ds(d["X_seq_test"], np.zeros(len(d["y_test"])),
                     CNN_BATCH, shuffle=False), verbose=1).flatten()
    cnn_pred = (cnn_prob > 0.5).astype(int)
    cnn_res  = evaluate("COA-CNN+BiLSTM", d["y_test"], cnn_pred, cnn_prob)
    best_t   = best_threshold(d["y_test"], cnn_prob)
    cnn_res["optimal_threshold"] = round(best_t,4)
    cnn_res["f1_at_optimal"] = round(float(f1_score(d["y_test"],(cnn_prob>best_t).astype(int))),4)
    ds_results.append(cnn_res)
    print(f"  COA-CNN: F1={cnn_res['F1-score']}  MCC={cnn_res['MCC']}")

    np.save(f"{r_dir}/cnn_coa_prob.npy", cnn_prob)
    json.dump(cnn_res, open(f"{r_dir}/cnn_coa_metrics.json","w"), indent=2)
    model.save(f"{m_dir}/coa_cnn_final.keras")
    del model; gc.collect()

    # Training curves
    pd.DataFrame(ds_results).to_csv(f"{r_dir}/coa_metrics.csv", index=False)

    all_results[ds] = {"ann": ann_res, "cnn": cnn_res}
    print(f"\n  Saved to {r_dir}/ and {m_dir}/")
    del d, tv, emb; gc.collect()

print("\n=== Final Training Complete ===")



FINAL TRAINING | AMAZON
  amazon: train=450,000 val=150,000 test=400,000 max_len=80
  COA-ANN   : H1=131 H2=115 LR=0.00240240  (val_F1=0.8621)
  COA-CNN   : F=256 LSTM=74 LR=0.00534078  (val_F1=0.8806)

  Training COA-ANN (batch=2048)...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/dev

I0000 00:00:1780323070.603410      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780323070.603429      69 cuda_dnn.cc:529] Loaded cuDNN version 91002


2110/2110 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8430 - loss: 0.3493
Epoch 1: val_loss improved from None to 0.25820, saving model to /kaggle/working/saved_models/amazon/coa_cnn_best.keras

Epoch 1: finished saving model to /kaggle/working/saved_models/amazon/coa_cnn_best.keras
2110/2110 ━━━━━━━━━━━━━━━━━━━━ 40s 17ms/step - accuracy: 0.8714 - loss: 0.3028 - val_accuracy: 0.8928 - val_loss: 0.2582
Epoch 2/30
2108/2110 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8947 - loss: 0.2562
Epoch 2: val_loss did not improve from 0.25820
2110/2110 ━━━━━━━━━━━━━━━━━━━━ 33s 16ms/step - accuracy: 0.8978 - loss: 0.2504 - val_accuracy: 0.8926 - val_loss: 0.2606
Epoch 3/30
2107/2110 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9049 - loss: 0.2336
Epoch 3: val_loss did not improve from 0.25820
2110/2110 ━━━━━━━━━━━━━━━━━━━━ 33s 16ms/step - accuracy: 0.9077 - loss: 0.2289 - val_accuracy: 0.8939 - val_loss: 0.2684
Epoch 4/30
2108/2110 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9119 - l

## 3. Summary

In [7]:
print("=== COA Model Results on Test Set ===\n")
for ds in DATASETS:
    ann = all_results[ds]["ann"]
    cnn = all_results[ds]["cnn"]
    print(f"{ds.upper()}")
    print(f"  COA-ANN        : F1={ann['F1-score']}  MCC={ann['MCC']}  "
          f"AUC={ann['AUC-ROC']}  OptThresh={ann['optimal_threshold']}")
    print(f"  COA-CNN+BiLSTM : F1={cnn['F1-score']}  MCC={cnn['MCC']}  "
          f"AUC={cnn['AUC-ROC']}  OptThresh={cnn['optimal_threshold']}")
    print()


=== COA Model Results on Test Set ===

AMAZON
  COA-ANN        : F1=0.866  MCC=0.7342  AUC=0.9411  OptThresh=0.4284
  COA-CNN+BiLSTM : F1=0.894  MCC=0.7842  AUC=0.9603  OptThresh=0.5008

YELP
  COA-ANN        : F1=0.8989  MCC=0.7987  AUC=0.9649  OptThresh=0.4334
  COA-CNN+BiLSTM : F1=0.9092  MCC=0.8137  AUC=0.9739  OptThresh=0.6108

TWITTER
  COA-ANN        : F1=0.7674  MCC=0.5387  AUC=0.8513  OptThresh=0.3482
  COA-CNN+BiLSTM : F1=0.7658  MCC=0.5378  AUC=0.8528  OptThresh=0.3746

